# Finetune Qwen3 with LLaMA Factory

Please use a **free** Tesla T4 Colab GPU to run this!

Project homepage: https://github.com/hiyouga/LLaMA-Factory

## Install Dependencies

In [1]:
%cd /kaggle/working/
import os
os.chdir('/kaggle/working/')

/kaggle/working


In [2]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 614, done.
remote: Counting objects: 100% (614/614), done.
remote: Compressing objects: 100% (455/455), done.
remote: Total 614 (delta 150), reused 381 (delta 101), pack-reused 0 (from 0)
Receiving objects: 100% (614/614), 5.24 MiB | 13.12 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/kaggle/working/LLaMA-Factory
assets/       docker/    LICENSE      pyproject.toml  requirements/  tests/
CITATION.cff  docs/      Makefile     README.md       scripts/       tests_v1/
data/         examples/  MANIFEST.in  README_zh.md    src/


In [3]:
!pip install -e .[torch,bitsandbytes]
!pip install -U bitsandbytes>=0.46.1
!pip install datasets
!pip install fsspec gcsfs

Obtaining file:///kaggle/working/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 46.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/1

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

### Check GPU environment

In [5]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: https://medium.com/mlearning-ai/training-yolov4-on-google-colab-316f8fff99c6")
print("cuda OK")

cuda OK


## Update Identity Dataset

In [6]:
import datasets
    
orig_ds = datasets.load_dataset("SoelMgd/Poker_Dataset", split="train")

README.md:   0%|          | 0.00/586 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet:   0%|          | 0.00/3.79M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/424k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/47178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5243 [00:00<?, ? examples/s]

In [7]:
import openai

client = openai.OpenAI(
    base_url='https://api.infomaniak.com/2/ai/48/openai/v1',
    api_key=user_secrets.get_secret("IK_KEY")
)

TEACHER_MODEL = 'qwen3'

In [8]:
features = datasets.Features({
    "instruction": datasets.Value('string'),
    "input": datasets.Value('string'),
    "output": datasets.Value('string'),
    "logprobs": [
        {"token": datasets.Value('string'), "logprob": datasets.Value('float')}
    ]
})

empty_feat = {
    "instruction": [],
    "input": [],
    "output": [],
    "logprobs": []
}

In [10]:
MAX_COMPLETION_TOKEN=200
TOP_LOGPROBS=1

def completion(i, temperature):
    question = orig_ds[i]["question"]
    messages = [
        {
            "role": "system",
            "content": "You are playing Texas Hold'em poker no-limit. You will be given the transcription of a game. As an answer, you will output your next action in upper-case and nothing else.",
        },
        {
            "role": "game",
            "content": question
        }
    ]
    
    completion = client.chat.completions.create(
        model=TEACHER_MODEL,
        messages=messages,
        logprobs=True,
        top_logprobs=TOP_LOGPROBS,
        max_completion_tokens=MAX_COMPLETION_TOKEN,
        temperature=0.3
    )
    return completion, messages

def parseCompletion(items, temperature, completion, messages):
    item = {
        "instruction": messages[0]['content'],
        "input": messages[1]['content'],
        "output": completion.choices[0].message.content,
        "logprobs": [{'token': lp.token, 'logprob': lp.logprob} for lp in completion.choices[0].logprobs.content]
    }
    items.append(item)

In [ ]:
import tqdm
import json
import os

# Try to load existing data
if os.path.exists('/kaggle/working/dataset_low_temp.json'):
    with open('/kaggle/working/dataset_low_temp.json', 'r') as f:
        items_lt = json.load(f)
    with open('/kaggle/working/dataset_high_temp.json', 'r') as f:
        items_ht = json.load(f)
    print(f"LOADED FROM DISK: {len(items_lt)} low-temp, {len(items_ht)} high-temp")
else:
    print("FETCHING DATA...")
    items_lt = []
    items_ht = []

    length = min(1000, len(orig_ds))
    
    temperature = 0.3
    for i in tqdm.trange(length//2):
        parseCompletion(items_lt, temperature, *completion(i, temperature))
        
    temperature = 0.9
    for i in tqdm.trange(length//2, length):
        parseCompletion(items_ht, temperature, *completion(i, temperature))

    print(f"DATA FETCHED: {len(items_lt)} low-temp, {len(items_ht)} high-temp")

In [ ]:
import json

# Save datasets as JSON for LLaMA Factory
with open('/kaggle/working/dataset_low_temp.json', 'w') as f:
    json.dump(items_lt, f, indent=2)

with open('/kaggle/working/dataset_high_temp.json', 'w') as f:
    json.dump(items_ht, f, indent=2)

print(f"Saved {len(items_lt)} low-temp and {len(items_ht)} high-temp examples")

# Register datasets in LLaMA Factory
dataset_info_path = '/kaggle/working/LLaMA-Factory/data/dataset_info.json'

with open(dataset_info_path, 'r') as f:
    dataset_info = json.load(f)

dataset_info['poker_low_temp'] = {
    "file_name": "/kaggle/working/dataset_low_temp.json",
    "formatting": "alpaca",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output"
    }
}

dataset_info['poker_high_temp'] = {
    "file_name": "/kaggle/working/dataset_high_temp.json",
    "formatting": "alpaca",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output"
    }
}

with open(dataset_info_path, 'w') as f:
    json.dump(dataset_info, f, indent=2)

print("✓ Datasets registered in LLaMA Factory")

## Fine-tune model via Command Line

It takes ~30min for training.

In [ ]:
%cd /kaggle/working/LLaMA-Factory/

!llamafactory-cli train \
    --stage sft \
    --do_train \
    --model_name_or_path unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit \
    --dataset poker_low_temp \
    --template qwen3_nothink \
    --finetuning_type lora \
    --lora_rank 16 \
    --lora_alpha 32 \
    --lora_target all \
    --output_dir /kaggle/working/outputs/stage1_low_temp \
    --overwrite_output_dir \
    --plot_loss \
    --trust_remote_code \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 5.0e-5 \
    --num_train_epochs 5.0 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.1 \
    --logging_steps 5 \
    --save_steps 50 \
    --cutoff_len 2048 \
    --preprocessing_num_workers 8 \
    --dataloader_num_workers 2 \
    --fp16 \
    --ddp_find_unused_parameters False \
    --report_to none

In [ ]:
%cd /kaggle/working/LLaMA-Factory/

!llamafactory-cli train \
    --stage sft \
    --do_train \
    --model_name_or_path unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit \
    --dataset poker_high_temp \
    --template qwen3_nothink \
    --finetuning_type lora \
    --lora_rank 16 \
    --lora_alpha 32 \
    --lora_target all \
    --adapter_name_or_path /kaggle/working/outputs/stage1_low_temp \
    --output_dir /kaggle/working/outputs/stage2_high_temp \
    --overwrite_output_dir \
    --plot_loss \
    --trust_remote_code \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 3.0e-5 \
    --num_train_epochs 5.0 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.1 \
    --logging_steps 5 \
    --save_steps 50 \
    --cutoff_len 2048 \
    --preprocessing_num_workers 8 \
    --dataloader_num_workers 2 \
    --fp16 \
    --ddp_find_unused_parameters False \
    --report_to none

## Infer the fine-tuned model

In [ ]:
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc

%cd /content/LLaMA-Factory/

args = dict(
  model_name_or_path="unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", # use bnb-4bit-quantized
)
chat_model = ChatModel(args)

messages = []
print("Welcome to the CLI application, use `clear` to remove the history, use `exit` to exit the application.")
while True:
  query = input("\nUser: ")
  if query.strip() == "exit":
    break
  if query.strip() == "clear":
    messages = []
    torch_gc()
    print("History has been removed.")
    continue

  messages.append({"role": "user", "content": query})
  print("Assistant: ", end="", flush=True)

  response = ""
  for new_text in chat_model.stream_chat(messages):
    print(new_text, end="", flush=True)
    response += new_text
  print()
  messages.append({"role": "assistant", "content": response})

torch_gc()

## Merge the LoRA adapter and optionally upload model

NOTE: the Colab free version has merely 12GB RAM, where merging LoRA of a 8B model needs at least 18GB RAM, thus you **cannot** perform it in the free version.

In [ ]:
!huggingface-cli login

In [ ]:
import json

args = dict(
  model_name_or_path="unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
  adapter_name_or_path="/content/drive/MyDrive/tp4_dasd/outputs/stage1",# load the saved LoRA adapters
  template="qwen3_nothink",                                        # same to the one in training
  finetuning_type="lora",                                   # same to the one in training
  export_dir="qwen3_lora_merged",                          # the path to save the merged model
  export_size=2,                                            # the file shard size (in GB) of the merged model
  export_device="cpu",                                      # the device used in export, can be chosen from `cpu` and `auto`
  # export_hub_model_id="your_id/your_model",               # the Hugging Face hub ID to upload model
)

json.dump(args, open("merge_qwen3.json", "w", encoding="utf-8"), indent=2)

%cd /content/LLaMA-Factory/

!llamafactory-cli export merge_qwen3.json